# Who has the ball? A walkthrough of the Athletica Intelligentia solution

This notebook explains, step by step, how we label every video frame of a football match with
**who has the ball**, and why our approach works. It is meant to be read top to bottom. Every
chart is produced from the data as you run it, and the functions it uses are imported from
`solution/run_inference.py`, the same code that produces our submission. What you read here is
what runs.

**Contents**

1. The problem
2. Setting up
3. How we are scored, and how the baseline does
4. Missing ball is not the same as dead ball
5. What does a dead ball look like?
6. Step 1: detecting dead balls
7. Step 2: filling the gaps
8. Putting it together
9. Limitations and open questions
10. Reproducing the results

## 1. The problem

We have one match, FC Barcelona vs Athletic Club, filmed at 25 frames per second: 199,319 frames.
For every frame we must answer one of three things:

| label | meaning |
|---|---|
| `H` | the home team (Barcelona) has the ball |
| `A` | the away team (Athletic) has the ball |
| `D` | **dead ball**: nobody has it, because play is stopped (ball out, foul, restart being set up) |

Only the two halves are graded (158,857 frames). Before kick-off, half-time and after the final
whistle are ignored.

**What we have to work with**

- **Tracking data.** For every frame, the position of every player the camera system can see, which
  team they are on, and the position of the ball *when it can see it*. The ball is missing on about
  41% of graded frames: it gets hidden behind players, flies out of shot, or leaves the pitch.
- **Smart Tagging.** An automatic list of match events produced from the same video, including 59
  *set pieces* (throw-ins, corners, goal kicks, free kicks, kick-offs) with their start times.

**The baseline** (`starter/baseline.py`) is deliberately simple. On each frame, the team of the
player closest to the ball has it. If the ball is not visible, say `D`. Then smooth the answer with
a majority vote over a sliding window, so it does not flicker every frame.

Our solution keeps this baseline and fixes its two biggest weaknesses with two extra steps. Each
step is motivated by what the data shows.

## 2. Setting up

We load the tracking data, the baseline's per-frame answer, the Smart Tagging set pieces, and the
public ground truth (the true labels for the first five minutes of the match). The tracking is
cached under `data/cache/` after the first run, so later runs start in seconds.

In [ ]:
import os, sys, json, warnings
import xml.etree.ElementTree as ET
from contextlib import contextmanager
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

ROOT = os.path.abspath(".")
while not os.path.exists(os.path.join(ROOT, "games.py")):   # the notebook may be opened from solution/
    ROOT = os.path.dirname(ROOT)
for p in (ROOT, os.path.join(ROOT, "solution"), os.path.join(ROOT, "evaluation")):
    if p not in sys.path:
        sys.path.insert(0, p)

import games
import possession_metrics as pm
import run_inference as ri                 # our solution
from starter import baseline as bl
from starter.baseline import HOME, AWAY, NOBODY

FPS = ri.FPS
paths = games.asset_paths()
PITCH = games.GAMES[games.GAME]["pitch"]
LENGTH, WIDTH = PITCH
BLOCKS = games.half_video_blocks(games.GAME)

C_HOME, C_AWAY, C_DEAD = "#0072B2", "#E69F00", "#BBBBBB"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# The baseline's answer before smoothing: HOME / AWAY for the nearest player's team, NOBODY when the ball is missing.
codes_raw = bl.build()
N = len(codes_raw)
graded = ri.graded_mask(N, BLOCKS)

# Ball positions and player speeds, cached after the first run.
cache = os.path.join(ROOT, "data", "cache", "tracking.npz")
if os.path.exists(cache):
    with np.load(cache) as z:
        ball_xy, player_speed = z["ball"], z["speed"]
else:
    ball_xy, player_speed = ri.load_tracking(paths["atd_raw"], len(bl.team_of_each_track(paths["atd_meta"])), PITCH)
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    with open(os.path.join(os.path.dirname(cache), ".gitignore"), "w") as handle:
        handle.write("*\n")                     # match data is never committed
    np.savez_compressed(cache, ball=ball_xy, speed=player_speed)

# Smart Tagging set pieces: start frame and type.
set_pieces = []
for inst in ET.parse(paths["smart_tagging"]).getroot().iter("instance"):
    if (inst.findtext("code") or "").strip().upper() == "SET PIECES":
        tags = {l.findtext("group"): l.findtext("text") for l in inst.iter("label")}
        set_pieces.append({"start": int(round(float(inst.findtext("start")) * FPS)),
                           "end": int(round(float(inst.findtext("end")) * FPS)),
                           "type": tags.get("Type", "?")})
set_pieces = pd.DataFrame(set_pieces).sort_values("start").reset_index(drop=True)
set_pieces["kind"] = set_pieces["type"].str.replace(r" \(.*\)", "", regex=True)
starts = ri.load_set_piece_starts(paths["smart_tagging"])

# The public ground truth: real labels for the first five minutes, X (not graded) elsewhere.
gt_codes = pm.label_codes(pm.read_label_csv(os.path.join(ROOT, "evaluation", "gt", "%s_possession_gt_public.csv" % games.GAME)))
gt_labels = np.array(pm.LABELS)[gt_codes]
PUBLIC = np.flatnonzero(gt_labels != "X")
pub = np.zeros(N, dtype=bool); pub[PUBLIC] = True
LO, HI = int(PUBLIC[0]), int(PUBLIC[-1])

print("%d frames, %d graded; ball visible on %.1f%% of graded frames" % (N, graded.sum(), 100 * np.mean(codes_raw[graded] != NOBODY)))
print("%d set pieces: %s" % (len(set_pieces), set_pieces.kind.value_counts().to_dict()))
print("public ground truth: frames %d-%d (%.0f s)" % (LO, HI, (HI - LO + 1) / FPS))

## 3. How we are scored, and how the baseline does

The challenge uses two headline numbers:

- **macroF1: right label on the right frame.** An F1 score is computed separately for `H`, `A` and
  `D`, then averaged. Because the three classes count equally, ignoring dead balls costs a full
  third of the score.
- **spellF1: right possessions.** This is the one the leaderboard ranks on. A *possession* is an
  uninterrupted run of `H` or of `A`. Each predicted possession is matched to a true one if they
  overlap enough in time. A dead-ball stretch is what separates two possessions, so getting `D`
  right is also what puts possession boundaries in the right place.

`score()` below runs the official scorer against the public five minutes and adds a few
whole-match statistics that need no ground truth.

> **A word of caution.** The public ground truth is only five minutes long, with 17 possessions.
> Missing a single possession moves spellF1 by about six points. We use these scores to check
> that each step *does what it is meant to*, not to fine-tune numbers. That is also why we always
> look at the whole-match statistics alongside them.

In [ ]:
def score(labels):
    r = pm.evaluate(gt_codes, np.where(labels == "H", 0, np.where(labels == "A", 1, 2)).astype(np.int8))
    return {"spellF1": r["spell_f1_mean"], "macroF1": r["macro_f1"],
            "recall_H": r["per_class"]["H"]["recall"], "recall_A": r["per_class"]["A"]["recall"],
            "precision_D": r["per_class"]["D"]["precision"], "recall_D": r["per_class"]["D"]["recall"],
            "possessions": r["n_pred_spells"], "D_%_public": 100 * np.mean(labels[PUBLIC] == "D"),
            "D_%_match": 100 * np.mean(labels[graded] == "D")}

SMOOTH = 50          # +/- 2 s majority vote: the challenge's published baseline
truth = pm.evaluate(gt_codes, gt_codes)
print("public ground truth: %d possessions, mean %.1f s, %.1f%% dead ball\n" % (
    truth["n_gt_spells"], truth["gt_mean_spell_duration_s"], 100 * np.mean(gt_labels[PUBLIC] == "D")))
pd.DataFrame([{"baseline": "no smoothing", **score(bl.smooth(codes_raw, 0))},
              {"baseline": "smoothed +/-%d" % SMOOTH, **score(bl.smooth(codes_raw, SMOOTH))}]).round(3)

The smoothed baseline gets the *team* right most of the time (high `recall_H`, `recall_A`) but
finds only about a fifth of the dead-ball frames (`recall_D`). Dead balls are its weak spot, and
they matter for both scores.

## 4. Missing ball is not the same as dead ball

The baseline's only notion of a dead ball is "the camera cannot see the ball". The public ground
truth lets us test that assumption directly.

In [ ]:
missing = codes_raw == NOBODY
dead_true = gt_labels == "D"
pd.DataFrame({
    "frames": [np.sum(pub & dead_true), np.sum(pub & dead_true & missing), np.sum(pub & dead_true & ~missing),
               np.sum(pub & missing), np.sum(pub & missing & dead_true), np.sum(pub & missing & ~dead_true)]},
    index=["truly dead", "  ... ball missing", "  ... ball VISIBLE",
           "ball missing", "  ... truly dead", "  ... truly in possession"])

The assumption fails in both directions:

1. **Two thirds of real dead-ball time has the ball visible.** For example, the ball is out of play
   but still tracked, or being placed for a restart. The baseline calls these frames possession.
2. **Almost half of the missing-ball frames are live play.** The ball is hidden behind players or in
   the air, and a team still has it. The baseline calls these frames dead.

Our solution has one step for each failure:

- **Step 1** detects dead balls from what the *players* and the *ball* are doing, whether or not the
  ball is visible.
- **Step 2** fills the remaining short missing-ball gaps with the team that clearly had the ball
  before and after.

## 5. What does a dead ball look like?

Before designing a detector, we measure. For every frame we compute a handful of simple,
interpretable variables:

| variable | meaning |
|---|---|
| `player_speed` | average speed of all tracked players (m/s), smoothed over one second |
| `ball_speed` | how fast the ball moves (m/s), from consecutive positions |
| `ball_line_dist` | distance from the ball to the nearest touchline or goal line (m); negative = outside the pitch |
| `last_line_dist` | `ball_line_dist` at the last moment the ball was seen (carried through gaps) |
| `ball_still_s` | how long the ball has been (almost) stationary (s) |
| `gap_s` | length of the missing-ball gap the frame sits in (s) |
| `sp_from_start_s` | seconds to/from the nearest Smart Tagging set-piece start |

In [ ]:
def run_length(mask):
    out = np.zeros(len(mask), dtype=np.int32)
    for s, e in zip(*ri.runs_of(mask)):
        out[s:e] = e - s
    return out

visible = ~np.isnan(ball_xy[:, 0])
ball_speed = pd.Series(np.r_[np.nan, np.hypot(*np.diff(ball_xy, axis=0).T)] * FPS) \
    .rolling(5, center=True, min_periods=3).median().to_numpy(copy=True)
ball_speed[~visible] = np.nan
x, y = ball_xy[:, 0], ball_xy[:, 1]
ball_line_dist = np.minimum.reduce([x, LENGTH - x, y, WIDTH - y])

nearest = starts[np.clip(np.searchsorted(starts, np.arange(N)), 0, len(starts) - 1)]
previous = starts[np.clip(np.searchsorted(starts, np.arange(N)) - 1, 0, len(starts) - 1)]
nearest = np.where(np.abs(np.arange(N) - previous) < np.abs(np.arange(N) - nearest), previous, nearest)

feat = pd.DataFrame({
    "ball_visible": visible,
    "player_speed": ri.smoothed_player_speed(player_speed),
    "ball_speed": ball_speed,
    "ball_line_dist": ball_line_dist,
    "last_line_dist": pd.Series(ball_line_dist).ffill().values,
    "ball_still_s": run_length(visible & (np.nan_to_num(ball_speed, nan=99) < 1.0)) / FPS,
    "gap_s": run_length(~visible) / FPS,
    "sp_from_start_s": (np.arange(N) - nearest) / FPS,
})
feat[graded].describe().T[["mean", "50%", "min", "max"]].round(2)

### 5a. Which variables separate dead from live?

For each variable we ask: if we pick one truly dead frame and one live frame at random, how often
does the dead frame have the higher value? This is the **AUC**. A value of 0.5 means the variable is
useless; values near 0 or 1 mean it separates the two well (near 0 means "lower when dead").
`strength` folds this onto 0.5-1 so the variables can be ranked.

In [ ]:
def auc(dead_values, live_values):
    a, b = dead_values[~np.isnan(dead_values)], live_values[~np.isnan(live_values)]
    ranks = pd.Series(np.r_[a, b]).rank().values
    return (ranks[:len(a)].sum() - len(a) * (len(a) + 1) / 2) / (len(a) * len(b))

f_pub = feat.iloc[PUBLIC].reset_index(drop=True)
is_dead = gt_labels[PUBLIC] == "D"
vis = f_pub.ball_visible.values
rows = []
for col in ["player_speed", "ball_line_dist", "last_line_dist", "ball_speed", "sp_from_start_s", "ball_still_s", "gap_s"]:
    v = f_pub[col].astype(float).values
    a = auc(v[is_dead & vis], v[~is_dead & vis]) if col != "gap_s" else auc(v[is_dead], v[~is_dead])
    rows.append({"variable": col, "AUC": a, "strength": max(a, 1 - a), "lower or higher when dead": "lower" if a < 0.5 else "higher",
                 "median when dead": np.nanmedian(v[is_dead]), "median when live": np.nanmedian(v[~is_dead])})
pd.DataFrame(rows).sort_values("strength", ascending=False).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, (col, lo, hi) in zip(axes, [("player_speed", 0, 5), ("ball_line_dist", -8, 34), ("ball_speed", 0, 25)]):
    bins = np.linspace(lo, hi, 50)
    for mask, colour, name in [(is_dead & vis, "#D55E00", "dead"), (~is_dead & vis, "#0072B2", "live")]:
        ax.hist(f_pub[col].values[mask].clip(lo, hi), bins=bins, density=True, histtype="step", lw=1.6, color=colour, label=name)
    ax.set_title(col); ax.set_yticks([])
axes[0].legend(frameon=False)
fig.suptitle("Public five minutes, ball visible: distribution when dead vs live", fontsize=10)
plt.tight_layout(); plt.show()

Two signals stand out:

- **Players slow down together.** When play stops, the average player speed roughly halves (about
  1.2 m/s against 2.3 m/s). This is the strongest single signal, and it does not need the ball at all.
- **Dead balls happen near the lines.** The ball is typically about 4 m from a line when play is
  dead, against about 13 m in live play. Balls go out over lines, and restarts are taken from them.

A stationary ball, perhaps surprisingly, tells us almost nothing (`ball_still_s`). During a
stoppage the ball is usually being carried, thrown back or rolled into place rather than sitting
still.

### 5b. Watching the dead balls happen

The same variables over the public five minutes. Grey bands are the true dead-ball stretches. Red
dashed lines are Smart Tagging set-piece starts.

In [ ]:
t = (np.arange(LO, HI + 1) - LO) / FPS
dead_s, dead_e = ri.runs_of(gt_labels[LO:HI + 1] == "D")
panels = [("player_speed", "mean player speed (m/s)", (0, 5)), ("ball_line_dist", "ball to nearest line (m)", (-8, 34)),
          ("ball_speed", "ball speed (m/s)", (0, 30)), ("ball_visible", "ball visible", (-0.1, 1.1))]
fig, axes = plt.subplots(len(panels), 1, figsize=(16, 7.5), sharex=True)
sp_in = set_pieces[(set_pieces.start >= LO) & (set_pieces.start <= HI)]
for ax, (col, label, ylim) in zip(axes, panels):
    ax.plot(t, feat[col].values[LO:HI + 1].astype(float), color="#222222", lw=0.8)
    for s, e in zip(dead_s, dead_e):
        ax.axvspan(s / FPS, e / FPS, color=C_DEAD, alpha=0.6, lw=0)
    for r in sp_in.itertuples():
        ax.axvline((r.start - LO) / FPS, color="#D55E00", ls="--", lw=1)
    ax.set_ylim(*ylim); ax.set_ylabel(label, fontsize=8)
axes[1].axhline(0, color="#D55E00", lw=0.6)
for r in sp_in.itertuples():
    axes[0].text((r.start - LO) / FPS, 4.9, " " + r.type, color="#D55E00", fontsize=8, va="top")
axes[-1].set_xlabel("seconds from kick-off")
plt.tight_layout(); plt.show()

Every grey band lines up with a drop in player speed. The last one shows the full story of a
typical stoppage: the ball crosses the goal line (distance below zero), it is placed about 4 m inside
the line for a goal kick, Smart Tagging marks the goal kick, and play resumes shortly after.

### 5c. What do the Smart Tagging set pieces mark?

It is tempting to treat each set-piece *interval* as dead ball. The chart below lines up all 59 set
pieces in the match on their start time (t = 0) and averages what happens around them.

In [ ]:
offsets = np.arange(-40 * FPS, 30 * FPS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6), sharex=True)
for kind, part in set_pieces.groupby("kind"):
    idx = part.start.values[:, None] + offsets[None, :]
    idx = idx[(idx[:, 0] >= 0) & (idx[:, -1] < N)]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        axes[0].plot(offsets / FPS, np.nanmean(feat.player_speed.values[idx], axis=0), lw=1.2, label="%s (n=%d)" % (kind, len(idx)))
    axes[1].plot(offsets / FPS, feat.ball_visible.values[idx].mean(axis=0), lw=1.2)
for ax, title in zip(axes, ["mean player speed (m/s)", "share of set pieces with the ball visible"]):
    ax.axvline(0, color="#222222", ls="--", lw=0.8); ax.set_title(title, fontsize=10); ax.set_xlabel("seconds from set-piece start")
axes[0].legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()
print("set-piece interval length: median %.0f s" % ((set_pieces.end - set_pieces.start).median() / FPS))

The start of a set piece marks the **restart**. Players are slowest right at t = 0 and accelerate
within seconds. Most of each 35-second interval is live play after the kick, so the interval as a
whole is not a dead ball. The dead ball sits *before* the start, from the moment the ball left play.

### 5d. When did the ball leave play?

We call a **ball exit** any moment the ball is seen outside the pitch, or vanishes within 2 m of a
line. If set pieces are usually preceded by an exit, then a dead ball can be defined as "from the
exit to the restart".

In [ ]:
exits = ri.ball_exits(ball_xy, graded, PITCH)
before = []
for s in set_pieces.start:
    prior = exits[(exits < s) & (exits >= s - 60 * FPS)]
    before.append((s - prior[-1]) / FPS if len(prior) else np.nan)
set_pieces["exit_before_s"] = before
followed = np.array([np.any((starts > e) & (starts <= e + 60 * FPS)) for e in exits])
print("%d ball exits in graded play; %.0f%% are followed by a tagged set piece within 60 s"
      % (len(exits), 100 * followed.mean()))
set_pieces[graded[set_pieces.start]].groupby("kind").agg(
    set_pieces=("start", "size"), with_exit_before=("exit_before_s", lambda s: int(s.notna().sum())),
    median_seconds_since_exit=("exit_before_s", "median")).round(1)

Most set pieces are preceded by an exit, typically 10-30 seconds earlier. That gap is the dead
ball. But only about 60% of exits are followed by a tagged set piece. Smart Tagging misses many
restarts (it lists only 26 throw-ins for a whole match), and some stoppages, such as fouls and
injuries, have neither an exit nor a set piece. The one signal present in **every** stoppage is the
players slowing down.

## 6. Step 1: detecting dead balls

Putting the evidence together gives our definition:

> **A dead ball is a stretch where the players slow down together, supported by evidence that play
> stopped.** It starts when the ball left play (if it did) and ends at the restart.

In detail (`dead_ball_v2` in `run_inference.py`):

1. Find every stretch where average player speed stays below **1.5 m/s for at least 3 s**.
2. Keep it if it is **anchored**:
   - a **ball exit** shortly before it (up to 15 s): the dead ball then starts at the exit;
   - a **set-piece start** inside it or just after (up to 5 s): the dead ball then ends at the restart.
3. Without an anchor, keep it only if it is **slower and longer** (below 1.2 m/s for at least 5 s).
   These are the fouls and injuries Smart Tagging does not list.
4. Mark those frames as `D`. They are forced to `D` *after* smoothing, so the majority vote cannot
   overrule them.

Every number above is a setting at the top of `run_inference.py`. Here is the function itself:

In [ ]:
import inspect
print(inspect.getsource(ri.dead_ball_v2))

In [ ]:
@contextmanager
def settings(**overrides):
    # Temporarily change settings in run_inference.py, e.g. settings(STOP_PLAYER_SPEED=1.2).
    old = {k: getattr(ri, k) for k in overrides}
    for k, v in overrides.items():
        setattr(ri, k, v)
    try:
        yield
    finally:
        for k, v in old.items():
            setattr(ri, k, v)


def run(pipeline, window=SMOOTH):
    codes, dead = ri.augment(codes_raw, ball_xy, player_speed, starts, BLOCKS, PITCH, pipeline=pipeline)
    return ri.finalize(codes, dead, window), dead

_, dead_mask = run({"dead_ball_v2": True})
hit = dead_mask & pub
print("dead-ball detector on its own, before any smoothing:")
pd.DataFrame([
    {"detector": "baseline (ball missing)", "precision_%": 100 * np.mean(dead_true[pub & missing]),
     "recall_%": 100 * np.sum(pub & missing & dead_true) / np.sum(pub & dead_true), "D_%_match": 100 * missing[graded].mean()},
    {"detector": "Step 1 (dead_ball_v2)", "precision_%": 100 * np.mean(dead_true[hit]),
     "recall_%": 100 * np.sum(hit & dead_true) / np.sum(pub & dead_true), "D_%_match": 100 * dead_mask[graded].mean()},
]).round(1)

*Precision* is how often a frame we call dead really is dead. *Recall* is how many of the truly
dead frames we find. The detector finds about two thirds of the dead time, and is right about
three times in four when it fires.

**Is it fragile?** A good rule should not depend on one lucky threshold. Below we vary the main
settings and score the full result (Step 1 + smoothing) each time.

In [ ]:
variants = [("as shipped", {})] + [("speed below %.1f m/s" % v, {"STOP_PLAYER_SPEED": v}) for v in (1.2, 1.8)] + [
    ("no unanchored slowdowns", {"USE_UNANCHORED": False}),
    ("restart = players speed up", {"RESTART_AT": "acceleration"})]
rows = [{"variant": "baseline", **score(bl.smooth(codes_raw, SMOOTH))}]
for name, overrides in variants:
    with settings(**overrides):
        rows.append({"variant": name, **score(run({"dead_ball_v2": True})[0])})
pd.DataFrame(rows).round(3)

- **The speed threshold barely matters** between 1.2 and 1.8 m/s, so the rule is not tuned to a
  lucky number.
- **Unanchored slowdowns matter a lot.** Without them, dead-ball recall drops sharply, because two of
  the four stoppages in the public window have no exit and no set piece.
- **Ending at the set-piece start beats waiting for players to speed up.** Players are still slow
  for a few seconds after the kick, by which time a team already has the ball.

## 7. Step 2: filling the gaps

With dead balls handled, a missing ball is much more likely to be a **hidden ball during play**:
the ball is behind a crowd of players, or in the air. If the same team has the ball just before
and just after the gap, it almost certainly had it throughout.

How the missing-ball gaps look over the whole match:

In [ ]:
gap_rows = ri.find_gaps(codes_raw, BLOCKS, ri.CONTEXT_FRAMES)
gaps = pd.DataFrame(gap_rows, columns=["start", "end", "length", "left", "right"])
gaps["seconds"] = gaps.length / FPS
known = (gaps.left != NOBODY) & (gaps.right != NOBODY)
gaps["either side"] = np.where(~known, "unknown", np.where(gaps.left == gaps.right, "same team", "different teams"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.8))
ax1.hist(gaps.seconds, bins=np.logspace(np.log10(0.04), np.log10(gaps.seconds.max() + 1), 40), color="#555555")
ax1.set_xscale("log"); ax1.set_xlabel("gap length (s, log scale)"); ax1.set_ylabel("gaps"); ax1.set_title("How long is the ball missing?")
by = gaps.assign(bucket=pd.cut(gaps.seconds, [0, 4, 8, 20, 30, np.inf], labels=["<4s", "4-8s", "8-20s", "20-30s", ">30s"])) \
    .pivot_table(index="bucket", columns="either side", values="length", aggfunc="sum", fill_value=0, observed=False) / FPS / 60
by.reindex(columns=["same team", "different teams", "unknown"], fill_value=0).plot(
    kind="bar", stacked=True, ax=ax2, color=["#009E73", "#D55E00", C_DEAD], width=0.8)
ax2.set_xlabel("gap length"); ax2.set_ylabel("minutes of missing ball"); ax2.set_title("Which team is on either side?")
ax2.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
gaps.groupby("either side").agg(gaps=("length", "size"), minutes=("seconds", lambda s: round(s.sum() / 60, 1)), median_s=("seconds", "median"))

Gaps are rarely short: most last 5-30 seconds. About two thirds of the missing time has the **same
team on both sides**.

The rule (`gap_fill` in `run_inference.py`):

- **Same team on both sides, gap up to 20 s** (2.5 x the baseline's 8 s window): fill with that team.
- **Different teams on either side**: the ball changed hands during the gap. Short gaps (up to 2 s)
  are split down the middle; longer ones are left alone.
- **Never fill across a dead ball** found in Step 1. That gap is a stoppage, not a hidden ball.

In [ ]:
print(inspect.getsource(ri.gap_fill))

In [ ]:
rows = [{"longest gap filled": "none (baseline)", **score(bl.smooth(codes_raw, SMOOTH))}]
for mult in (1, 2.5, 3.75):
    with settings(GAP_FILL_MULTIPLIER=mult):
        rows.append({"longest gap filled": "%g s" % (ri.BASE_WINDOW_FRAMES * mult / FPS), **score(run({"gap_fill": True})[0])})
pd.DataFrame(rows).round(3)

On its own, gap filling makes possessions longer and fewer (closer to the true 17) and lifts
spellF1. The public window happens to contain no gaps between 8 and 30 s long, so these settings
score the same there. The whole-match `D_%_match` column shows that they do differ, and 20 s is a
middle ground.

On its own, gap filling *lowers* dead-ball recall, because it fills some real stoppages. That is
why it runs after Step 1, which protects dead balls from being filled.

## 8. Putting it together

The full pipeline:

1. **Baseline:** team of the nearest player when the ball is visible.
2. **Step 1:** mark dead balls.
3. **Step 2:** fill hidden-ball gaps, never across a dead ball.
4. **Smooth** with the baseline's majority vote.
5. **Force** the Step 1 frames to `D`.

In [ ]:
configs = [("baseline", {}), ("Step 1 only", {"dead_ball_v2": True}), ("Step 2 only", {"gap_fill": True}),
           ("Step 1 + Step 2 (our solution)", {"dead_ball_v2": True, "gap_fill": True})]
results = pd.DataFrame([{"pipeline": name, **score(run(flags)[0] if flags else bl.smooth(codes_raw, SMOOTH))}
                        for name, flags in configs])
results.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 2.8), sharey=True)
y = np.arange(len(results))[::-1]
for ax, col in zip(axes, ["spellF1", "macroF1"]):
    ax.barh(y, results[col], color=["#999999", "#56B4E9", "#56B4E9", "#0072B2"])
    for yi, v in zip(y, results[col]):
        ax.text(v, yi, " %.3f" % v, va="center", fontsize=9)
    ax.set_title("%s (public five minutes, smoothing +/-%d)" % (col, SMOOTH), fontsize=10)
axes[0].set_yticks(y); axes[0].set_yticklabels(results.pipeline)
plt.tight_layout(); plt.show()

The two steps help in different ways and combine well. Step 1 is responsible for almost all of
the gain in **macroF1**, because dead balls go from mostly missed to mostly found. On **spellF1**
each step helps a little on its own, and together they help much more. Step 1 puts the
boundaries between possessions in the right place, and Step 2 stops possessions breaking up
every time the ball is hidden.

The timeline shows the whole public window, labelled by the baseline, by our solution, and by the
ground truth:

In [ ]:
def strip(ax, rows, names):
    lookup = {"H": 0, "A": 1, "D": 2}
    image = np.array([[lookup.get(v, 2) for v in r[LO:HI + 1]] for r in rows])
    ax.imshow(image, aspect="auto", interpolation="nearest", cmap=ListedColormap([C_HOME, C_AWAY, C_DEAD]),
              vmin=0, vmax=2, extent=[0, (HI - LO + 1) / FPS, len(rows), 0])
    ax.set_yticks(np.arange(len(rows)) + 0.5); ax.set_yticklabels(names); ax.set_xlabel("seconds from kick-off")
    for k in range(1, len(rows)):
        ax.axhline(k, color="white", lw=2)

ours, _ = run({"dead_ball_v2": True, "gap_fill": True})
fig, ax = plt.subplots(figsize=(16, 2.6))
strip(ax, [bl.smooth(codes_raw, SMOOTH), ours, gt_labels], ["baseline", "our solution", "ground truth"])
ax.set_title("blue = Barcelona (H), orange = Athletic (A), grey = dead ball (D)", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

**Across smoothing windows.** `run_inference.py` writes one submission per smoothing window. Our
solution beats the baseline on both metrics at every window:

In [ ]:
rows = []
for w in ri.windows:
    b, o = score(bl.smooth(codes_raw, w)), score(run({"dead_ball_v2": True, "gap_fill": True}, w)[0])
    rows.append({"window": w, "spellF1 baseline": b["spellF1"], "spellF1 ours": o["spellF1"],
                 "macroF1 baseline": b["macroF1"], "macroF1 ours": o["macroF1"],
                 "recall_H baseline": b["recall_H"], "recall_H ours": o["recall_H"]})
pd.DataFrame(rows).set_index("window").round(3)

## 9. Limitations and open questions

**Five minutes is a small sample.** Seventeen possessions cannot reliably separate two good
settings. The best-looking window on the public data is not necessarily the best over the full
match; the challenge's own documentation shows this happening to the baseline.

**How much of the match is dead?** Our solution labels about 46% of the whole match as dead ball,
against 20% in the public five minutes. A typical professional match has only 52-55 minutes of real
playing time, so 40-45% dead time is plausible, and the opening minutes had few stoppages. It cannot
be confirmed without the full ground truth.

**Food for thought: we lose a little home-team recall.** At every smoothing window, `recall_H` is
slightly lower than the baseline's. The table below shows where the true `H` frames go:

In [ ]:
base, ours = bl.smooth(codes_raw, SMOOTH), run({"dead_ball_v2": True, "gap_fill": True})[0]
h = pub & (gt_labels == "H")
pd.DataFrame({"baseline": [np.sum(base[h] == k) for k in "HAD"], "our solution": [np.sum(ours[h] == k) for k in "HAD"]},
             index=["true H predicted H", "true H predicted A", "true H predicted D"])

The loss comes almost entirely from Barcelona possession being called **dead**: frames the baseline
gave to `H` that Step 1 now marks `D`. One explanation is that Barcelona's slow, patient build-up,
with players barely moving while the ball circulates at the back, looks like a stoppage to a rule
built on player speed. If so, an extra signal is needed to tell "slow play" from "stopped play",
such as whether the ball keeps moving between teammates. Alternatively, some of these frames may be
genuinely ambiguous moments around restarts. Either way it is a good place to look next.

**Other known weaknesses**

- A ball that vanishes near a line is not always out of play. One false exit in the public window
  creates a wrong 10-second dead ball.
- Smart Tagging is automatic and incomplete. Missing restarts fall back on the slowdown rule.

## 10. Reproducing the results

Everything above is produced by one script. From the repository root:

```bash
uv run python solution/run_inference.py
```

It will:

1. compute the baseline's nearest-player answer with the challenge's own code;
2. apply Step 1 and Step 2;
3. write one submission per smoothing window to
   `evaluation/results/athletica_intelligentia_smooth_<window>.csv`;
4. score every submission with the official `evaluation/run_evaluation.py`, writing
   `evaluation/scoreboard.csv` and a JSON report per submission to `evaluation/reports/`;
5. save the exact settings used to `evaluation/reports/athletica_intelligentia_config.json`;
6. print baseline vs our solution side by side.

To start again from a clean slate:

```bash
./solution/clean.sh
```

To experiment, change a setting at the top of `solution/run_inference.py` (for example
`STOP_PLAYER_SPEED` or `GAP_FILL_MULTIPLIER`), or switch a step off in `PIPELINE`. Re-run this
notebook to see the effect, then re-run the script to produce new submissions.

The match data is gated and is not part of this repository. See `DATA.md` for how to obtain it.